# Build Your Own Tapis App with dapi

A Tapis app is two files: `app.json`, the definition Tapis registers (inputs, parameters, resources), and `tapisjob_app.sh`, the wrapper that runs on the compute node. Registering one requires no administrator. You own the record, the wrapper lives in your MyData, and nobody else sees the app until you share it.

This notebook builds one end to end: scaffold the files from a dapi template, edit the definition, register it, submit a job against it, and read the results. The [apps documentation](https://designsafe-ci.github.io/dapi/apps) covers every field of both files.

In [ ]:
%pip install --quiet --upgrade dapi

**Restart the kernel once after the install**, then run from the next cell.

In [1]:
from pathlib import Path

from dapi import DSClient

ds = DSClient()

# DesignSafe JupyterHub mounts your MyData at ~/MyData; scratch goes there
# since community folders are read-only. Anywhere else, write beside the
# notebook.
mydata = Path.home() / "MyData"
work_root = mydata if mydata.is_dir() else Path.cwd()

/Users/krishna/dev/DesignSafe/Dapi-Tapis/dapi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Authentication successful.


TMS credentials ready: frontera, stampede3


## Create the app files

dapi ships two templates. `zip` runs any shell command on the staged inputs; it is the skeleton to edit into a fully custom app. `container` runs any container image (see the [containers example](https://designsafe-ci.github.io/dapi/containers)). We build on `zip`.

In [2]:
import shutil

print("templates:", ds.apps.templates())

shutil.rmtree(work_root / "my-first-app", ignore_errors=True)  # rerun-safe
app_dir = ds.apps.new("my-first-app", target_dir=str(work_root), template="zip")
print("created:", app_dir)

templates: ['container', 'zip']
Created app 'my-first-app' from template 'zip' at /Users/krishna/MyData/my-first-app


created: /Users/krishna/MyData/my-first-app


## App Definition - app.json

`app.json` is the contract: the file inputs a job must provide, the parameters it may set (`envVariables`), and the default resources. Everything here is what `ds.jobs.generate` and the portal read.

In [3]:
print((work_root / "my-first-app" / "app.json").read_text())

{
  "id": "my-first-app",
  "version": "0.1.0",
  "description": "Custom app: runs COMMAND on the staged input directory, with optional TACC modules. Edit tapisjob_app.sh for anything beyond a single command.",
  "runtime": "ZIP",
  "runtimeVersion": null,
  "containerImage": "__SET_BY_DEPLOY__",
  "jobType": "BATCH",
  "strictFileInputs": true,
  "jobAttributes": {
    "execSystemId": "stampede3",
    "execSystemLogicalQueue": "skx",
    "execSystemExecDir": "${JobWorkingDir}",
    "execSystemInputDir": "${JobWorkingDir}",
    "execSystemOutputDir": "${JobWorkingDir}",
    "archiveSystemId": "designsafe.storage.default",
    "archiveSystemDir": "${JobOwner}/tapis-jobs-archive/${JobCreateDate}/${JobName}-${JobUUID}",
    "archiveOnAppError": true,
    "isMpi": false,
    "nodeCount": 1,
    "coresPerNode": 48,
    "maxMinutes": 60,
    "fileInputs": [
      {
        "name": "Input Directory",
        "description": "Directory staged to the job; COMMAND runs inside it and outputs writt

## Wrapper - tapisjob_app.sh

The wrapper is what SLURM executes. The template loads any requested modules (with `set -u` relaxed, since Lmod hooks are not nounset-clean), changes into the staged input directory, and runs `COMMAND`. Replace that last section with your own launch logic when one command is not enough.

In [4]:
print((work_root / "my-first-app" / "tapisjob_app.sh").read_text())

#!/bin/bash
# my-first-app: EDIT THIS WRAPPER. It runs on the compute node after Tapis
# stages the job's Input Directory. The two job parameters are
#
#   COMMAND        shell command, run inside the staged input directory
#   EXTRA_MODULES  comma-separated TACC modules to load first (optional)
#
# Outputs written to the input directory are archived when the job ends.
# Replace the COMMAND line at the bottom with your own launch logic when
# a single command is not enough (MPI launchers, staging, post steps).
set -euo pipefail
set -x

INPUTSCRIPT="${1:-}"  # unused; kept for parity with python-s3 calling convention

: "${COMMAND:?COMMAND is required}"

# Lmod modulefiles and their completion hooks are not nounset-clean,
# so module loads run with set -u relaxed.
if [[ -n "${EXTRA_MODULES:-}" ]]; then
    IFS=',' read -ra MODS <<< "${EXTRA_MODULES}"
    for mod in "${MODS[@]}"; do
        mod="$(echo "${mod}" | xargs)"
        if [[ -n "${mod}" ]]; then
            set +u
            m

## Edit the definition

Normally you edit the files in an editor; here we adjust them programmatically. We describe the app and shrink the defaults to match a small serial analysis.

In [5]:
import json

spec = json.load(open(work_root / "my-first-app" / "app.json"))
spec["description"] = "Summarizes a list of peak ground accelerations."
spec["jobAttributes"]["execSystemLogicalQueue"] = "skx-dev"
spec["jobAttributes"]["coresPerNode"] = 1
spec["jobAttributes"]["maxMinutes"] = 10
json.dump(spec, open(work_root / "my-first-app" / "app.json", "w"), indent=2)

print(
    "queue:",
    spec["jobAttributes"]["execSystemLogicalQueue"],
    "| cores:",
    spec["jobAttributes"]["coresPerNode"],
    "| minutes:",
    spec["jobAttributes"]["maxMinutes"],
)

queue: skx-dev | cores: 1 | minutes: 10


## Deploy

`deploy()` zips the wrapper, uploads it to your MyData, points the definition's `containerImage` at the zip, and registers the version under your ownership. Rerunning it with the same version updates the app in place, so the edit, deploy, submit loop is fast.

In [6]:
result = ds.apps.deploy(str(work_root / "my-first-app"))
result

Updated existing app my-first-app v0.1.0


{'app_id': 'my-first-app',
 'version': '0.1.0',
 'container_image': 'tapis://designsafe.storage.default/kks32/apps/my-first-app/0.1.0/my-first-app.zip'}

In [7]:
app = ds.tapis.apps.getAppLatestVersion(appId="my-first-app")
print(app.id, app.version, "| owner:", app.owner)

my-first-app 0.1.0 | owner: kks32


## Stage the analysis inputs

The app runs `COMMAND` inside the staged input directory, so the analysis script and its data travel with each job, not with the app. This one summarizes peak ground accelerations using only the standard library, so it needs no modules.

In [8]:
analyze = """
import json
import sys

values = [float(x) for x in open(sys.argv[1]).read().split(",")]
summary = {
    "count": len(values),
    "max_pga_g": max(values),
    "mean_pga_g": sum(values) / len(values),
}
json.dump(summary, open("summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))
"""
work_dir = work_root / "custom-app-inputs"
work_dir.mkdir(parents=True, exist_ok=True)
(work_dir / "analyze.py").write_text(analyze)
(work_dir / "example_input.txt").write_text("0.12,0.34,0.08,0.51,0.27,0.19,0.44")

inputs_uri = (
    "tapis://designsafe.storage.default/"
    + ds.tapis.username
    + "/dapi-demo/custom-app-inputs"
)
ds.files.upload(str(work_dir / "analyze.py"), inputs_uri + "/analyze.py")
ds.files.upload(str(work_dir / "example_input.txt"), inputs_uri + "/example_input.txt")
print("inputs at", inputs_uri)

inputs at tapis://designsafe.storage.default/kks32/dapi-demo/custom-app-inputs


## Submit a job against your app

Your `envVariables` are the job's parameters. `COMMAND` is required; `EXTRA_MODULES` is there when the analysis needs TACC software (e.g. `"opensees"`).

In [9]:
allocation = "DS-Portal-SPARC2026"  # <-- replace with your allocation

job = {
    "name": "custom-app-demo",
    "appId": "my-first-app",
    "appVersion": result["version"],
    "execSystemLogicalQueue": "skx-dev",
    "nodeCount": 1,
    "coresPerNode": 1,
    "maxMinutes": 10,
    "fileInputs": [{"name": "Input Directory", "sourceUrl": inputs_uri}],
    "parameterSet": {
        "envVariables": [
            {"key": "COMMAND", "value": "python3 analyze.py example_input.txt"},
        ],
        "schedulerOptions": [{"name": "TACC Allocation", "arg": "-A " + allocation}],
    },
}
submitted = ds.jobs.submit(job)
final = submitted.monitor(interval=15, timeout_minutes=45)
print("final status:", final)

Job submitted successfully. UUID: 66880112-7e32-4d06-9a65-17fd8ce31d1e-007



Monitoring Job: 66880112-7e32-4d06-9a65-17fd8ce31d1e-007


Waiting for job to start: 0 checks [00:00, ? checks/s]

Waiting for job to start: 0 checks [00:00, ? checks/s, Status: PENDING]

Waiting for job to start: 1 checks [00:15, 15.14s/ checks, Status: PENDING]

...

Monitoring job (Status: ARCHIVING):   3%|▎         | 6/180 [01:15<40:21, 13.92s/ checks]

Monitoring job (Status: ARCHIVING):   3%|▎         | 6/180 [01:30<40:21, 13.92s/ checks]

Monitoring job (Status: ARCHIVING): 100%|██████████| 180/180 [01:30<00:00, 13.92s/ checks]

Monitoring job (Status: ARCHIVING): 100%|██████████| 180/180 [01:30<00:00,  1.98 checks/s]

	Status: FINISHED
final status: FINISHED


## Read the outputs

Everything the command wrote into the input directory is archived, alongside `tapisjob.out` with the wrapper's log.

In [10]:
submitted.print_runtime_summary()
print(submitted.get_output_content("inputDirectory/summary.json"))


Runtime Summary
---------------


QUEUED  time: 00:00:01
RUNNING time: 00:00:35
TOTAL   time: 00:02:35
---------------


{
  "count": 7,
  "max_pga_g": 0.51,
  "mean_pga_g": 0.2785714285714286
}


## Iterate, share, clean up

Edit either file and run `deploy()` again; the same version updates in place. Bump `version` in `app.json` when collaborators depend on the current one. Share the app and collaborators submit against it with their own allocations:

```python
ds.tapis.apps.shareApp(appId="my-first-app", users=["collaborator1"])
```

Remove the demo app when you are done experimenting:

```python
ds.tapis.apps.deleteApp(appId="my-first-app")
```